# GNN-PCNA — MD Results Wormhole

Runs analysis on the completed 100ns trajectory from Google Drive and pushes results directly to GitHub.  
**You do NOT need to download the trajectory locally.**

**Before running:**
- Make sure the trajectory is saved at `GNN_PCNA_MD/1W60_production.dcd` in your Google Drive
- Add `GH_TOKEN` to Colab Secrets (key icon, left sidebar) — needs repo write access
- Runtime → GPU (T4) — needed for MDAnalysis alignment speed

In [ ]:
# Cell 1: Install deps + mount Drive
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'MDAnalysis', 'numpy', 'scipy'], check=True)

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Find trajectory
DRIVE_DIR = Path('/content/drive/MyDrive/GNN_PCNA_MD')
TRAJ  = DRIVE_DIR / '1W60_production.dcd'
TOP   = DRIVE_DIR / '1W60_topology.pdb'

# Fallback topology locations
if not TOP.exists():
    TOP = DRIVE_DIR / '1W60_solvated.pdb'

assert TRAJ.exists(), f'Trajectory not found at {TRAJ}'
assert TOP.exists(),  f'Topology not found at {TOP}'

traj_gb = TRAJ.stat().st_size / 1e9
print(f'Trajectory: {TRAJ}  ({traj_gb:.1f} GB)')
print(f'Topology:   {TOP}')

In [ ]:
# Cell 2: Clone repo (for scripts + output paths)
import subprocess, os
from google.colab import userdata

token = userdata.get('GH_TOKEN')
REPO_URL = f'https://{token}@github.com/Reshwant-Borra/GNN_PCNA.git'

if not os.path.exists('/content/GNN_PCNA'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/GNN_PCNA'], check=True)
else:
    subprocess.run(['git', '-C', '/content/GNN_PCNA', 'pull'], check=True)

os.chdir('/content/GNN_PCNA')
subprocess.run(['git', 'config', 'user.email', 'advay.awesomer@gmail.com'], check=True)
subprocess.run(['git', 'config', 'user.name', 'Advay'], check=True)
print('Repo ready')

In [ ]:
# Cell 3: Run full MD analysis on the Drive trajectory
import subprocess, sys
from pathlib import Path

TRAJ = Path('/content/drive/MyDrive/GNN_PCNA_MD/1W60_production.dcd')
TOP  = Path('/content/drive/MyDrive/GNN_PCNA_MD/1W60_topology.pdb')
if not TOP.exists():
    TOP = Path('/content/drive/MyDrive/GNN_PCNA_MD/1W60_solvated.pdb')
if not TOP.exists():
    TOP = Path('/content/GNN_PCNA/data/md/1W60_solvated.pdb')

print(f'Running analysis on {TRAJ.stat().st_size/1e9:.1f} GB trajectory...')
print(f'Topology: {TOP}')

result = subprocess.run([
    sys.executable, 'scripts/run_md_analysis.py',
    '--top',    str(TOP),
    '--traj',   str(TRAJ),
    '--stride', '10',
    '--out_dir', 'data/results',
], capture_output=False)

print('Return code:', result.returncode)
assert result.returncode == 0, 'MD analysis failed — check output above'

In [ ]:
# Cell 4: Print the key numbers
import json
from pathlib import Path

rmsf = json.loads(Path('data/results/md_rmsf_1W60.json').read_text())
comp = json.loads(Path('data/results/md_apo_comparison.json').read_text())
vol  = json.loads(Path('data/results/md_pocket_volume.json').read_text())

print('=== 100ns MD RESULTS ===')
print(f'Frames analysed:      {rmsf["n_frames"]}')
print(f'Simulation time:      {rmsf["sim_time_ns"]:.1f} ns')
print(f'Pocket RMSF:          {rmsf["pocket_rmsf_angstrom"]:.3f} A')
print(f'Background RMSF:      {rmsf["background_rmsf_angstrom"]:.3f} A')
print(f'Fold-change (p/bg):   {rmsf["fold_change_pocket_vs_bg"]:.3f}')
print()
print('Apo comparison:')
for k,v in comp.items():
    if not isinstance(v, (dict, list)):
        print(f'  {k}: {v}')
print()
print('Pocket volume:')
for k,v in vol.items():
    if not isinstance(v, (dict, list)):
        print(f'  {k}: {v}')

# Flag the result
fc = rmsf['fold_change_pocket_vs_bg']
if fc > 1.1:
    print(f'\n>>> POSITIVE: pocket more flexible than background (fold-change {fc:.3f})')
    print('    MD SUPPORTS cryptic pocket opening at AOH1996 site.')
elif fc < 0.9:
    print(f'\n>>> RIGID: pocket less flexible than background (fold-change {fc:.3f})')
    print('    Pocket is buried/rigid in apo state — consistent with closed cryptic site.')
    print('    Check for transient opening events in volume time series.')
else:
    print(f'\n>>> NEUTRAL: fold-change {fc:.3f} — inconclusive from RMSF alone.')
    print('    Check DCCM and volume time series for correlated motion.')

In [ ]:
# Cell 5: Regenerate figures
import subprocess, sys
result = subprocess.run([sys.executable, 'scripts/make_md_figures.py'], capture_output=False)
print('Figures done:', result.returncode == 0)

# Show inline
from IPython.display import Image, display
from pathlib import Path
for fig in ['fig4a_md_rmsf.png', 'fig4b_md_pocket_vol.png', 'fig4c_md_dccm.png']:
    p = Path(f'data/results/{fig}')
    if p.exists():
        print(f'\n--- {fig} ---')
        display(Image(str(p)))

In [ ]:
# Cell 6: Rebuild paper with MD results
import subprocess, sys
result = subprocess.run([sys.executable, 'scripts/build_paper_docx.py'], capture_output=False)
print('Paper built:', result.returncode == 0)

from pathlib import Path
papers = sorted(Path('docs').glob('*.docx'))
print('Available docs:')
for p in papers:
    print(f'  {p.name}  ({p.stat().st_size/1e3:.0f} KB)')

In [ ]:
# Cell 7: Push everything to GitHub
import subprocess, shutil
from pathlib import Path
from google.colab import userdata

# Copy paper to Drive for download
DRIVE_DIR = Path('/content/drive/MyDrive/GNN_PCNA_MD')
papers = sorted(Path('docs').glob('*.docx'), key=lambda p: p.stat().st_mtime)
if papers:
    shutil.copy2(papers[-1], DRIVE_DIR / papers[-1].name)
    print(f'Paper saved to Drive: {papers[-1].name}')

# Stage results
files_to_push = [
    'data/results/md_rmsf_1W60.json',
    'data/results/md_apo_comparison.json',
    'data/results/md_pocket_volume.json',
    'data/results/md_dccm_1W60.npy',
    'data/results/fig4a_md_rmsf.png',
    'data/results/fig4b_md_pocket_vol.png',
    'data/results/fig4c_md_dccm.png',
    'docs/',
]

subprocess.run(['git', 'add'] + files_to_push, check=True)

# Read key number for commit message
import json
rmsf = json.loads(Path('data/results/md_rmsf_1W60.json').read_text())
fc   = rmsf['fold_change_pocket_vs_bg']
ns   = rmsf['sim_time_ns']
pocket_rmsf = rmsf['pocket_rmsf_angstrom']
bg_rmsf     = rmsf['background_rmsf_angstrom']

msg = f"""feat: 100ns MD results — fold-change {fc:.3f}, pocket RMSF {pocket_rmsf:.3f}A

Full {ns:.0f}ns CHARMM36m production run on apo PCNA (1W60).
AOH1996 pocket: RMSF {pocket_rmsf:.3f}A  Background: {bg_rmsf:.3f}A
Fold-change (pocket/bg): {fc:.3f}

Updated paper with MD section 3.8 results.

Co-Authored-By: Claude Sonnet 4.6 <noreply@anthropic.com>"""

result = subprocess.run(['git', 'commit', '-m', msg])
if result.returncode == 0:
    subprocess.run(['git', 'push', 'origin', 'main'], check=True)
    print('Pushed to GitHub')
else:
    print('Nothing new to commit')

In [ ]:
# Cell 8: Download paper directly from Colab
from google.colab import files
from pathlib import Path

papers = sorted(Path('docs').glob('*.docx'), key=lambda p: p.stat().st_mtime)
if papers:
    latest = papers[-1]
    print(f'Downloading: {latest.name}')
    files.download(str(latest))
else:
    print('No paper found')

## After running

Results are now on GitHub. Pull locally:
```bash
git pull
```

Paper is also saved to `GNN_PCNA_MD/` in your Google Drive and downloaded directly.

### Interpreting fold-change
- **> 1.1** → pocket is more flexible than background → strong MD support for opening
- **0.9–1.1** → neutral → check DCCM for correlated motion, check volume for transient events  
- **< 0.9** → pocket is rigid in apo state → consistent with closed cryptic site, transient opening possible at longer timescales

The preliminary 6.36ns run showed fold-change **0.832** (rigid). The full 100ns run may show different dynamics.